# PyCRMKit 1.2.0 — 01. Core CRM Foundations
Ce notebook explore et valide les piliers fondamentaux de **PyCRMKit** en mémoire (`CRM.memory()`) :
- **Contacts** : création, mise à jour, emails normalisés, téléphones
- **Organisations** : entités B2B, domaines et adresses
- **Relations** : liens typés Contact ↔ Organisation avec rôles et contact principal (`is_primary`)
- **Tags & Custom Fields** : schéma typé et valeurs personnalisées
- **Activités** : réunions, appels, notes horodatées avec participants
- **Tâches & Calendrier Métier** : calcul d'échéances ouvrées (`BusinessCalendar`), cycle de vie
- **Timeline** : journal chronologique unifié par entité

## 1. Initialisation de l'environnement & Facade CRM
Nous initialisons `CRM.memory()` avec un contexte d'acteur et un identifiant de corrélation.

In [ ]:
import sys
from pathlib import Path

# Résolution automatique du chemin local pour importer le package du workspace
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from pycrmkit import CRM, __version__
from pycrmkit.core.references import EntityReference

print(f"✅ PyCRMKit version chargée : {__version__}")

# Création du CRM in-memory avec contexte d'audit
crm = CRM.memory().with_context(
    actor_id="notebook-admin",
    correlation_id="core-foundations-session-001"
)

print(f"CRM context : actor={crm.context.actor_id}, correlation={crm.context.correlation_id}")

## 2. Organisations & Contacts
Création d'une organisation cliente et d'un contact associé avec email normalisé.

In [ ]:
from pycrmkit.contacts import ContactEmail, ContactPhone

# 1. Création de l'Organisation
org = crm.organizations.create(
    legal_name="Institut Curie SAS",
    trading_name="Curie Research Labs",
    source="notebook-exploration",
)
print(f"🏢 Organisation créée : {org.legal_name} (ID: {org.id})")

# 2. Création du Contact
contact = crm.contacts.create(
    first_name="Marie",
    last_name="Curie",
    source="referral",
    emails=(ContactEmail("Marie.Curie@Institut-Curie.org", is_primary=True),),
    phones=(ContactPhone("+33 1 42 34 60 00", is_primary=True),),
)
print(f"👤 Contact créé : {contact.display_name}")
print(f"📧 Email normalisé automatiquement : {contact.emails[0].normalized}")

## 3. Relations Contact ↔ Organisation
PyCRMKit modélise les liaisons professionnelles via des `Relationship` typées avec rôle et drapeau de contact principal.

In [ ]:
from pycrmkit.relationships import RelationshipEndpoint, RelationshipType

rel = crm.relationships.create(
    source=RelationshipEndpoint.contact(contact.id),
    target=RelationshipEndpoint.organization(org.id),
    relationship_type=RelationshipType("employee_of"),
    role="Directrice de Recherche",
    is_primary=True,
)

print(f"🔗 Relation établie : {contact.display_name} -> {rel.role} chez {org.legal_name}")
print(f"   Contact principal ? {rel.is_primary}")

## 4. Tags & Custom Fields Typés
Définition d'un champ personnalisé sur l'entité Contact et affectation de tags.

In [ ]:
from pycrmkit.custom_fields import CustomFieldType

# 1. Tags
vip_tag = crm.tags.create(name="Key Account")
crm.tags.assign(tag_id=vip_tag.id, entity=EntityReference("contact", contact.id))
print(f"🏷️ Tag assigné : {vip_tag.name}")

# 2. Custom Field Definition & Value
field_def = crm.custom_fields.define(
    key="security_clearance",
    label="Habilitation Sécurité",
    field_type=CustomFieldType.STRING,
    applies_to=("contact",),
)

field_val = crm.custom_fields.set_value(
    definition_id=field_def.id,
    entity=EntityReference("contact", contact.id),
    value="Niveau Confidentiel Défense",
)

print(f"⚙️ Custom Field défini : {field_def.label} (clé: {field_def.key})")
print(f"   Valeur enregistrée : {field_val.value}")

## 5. Activités & Tâches avec Calendrier Métier
Enregistrement d'un rendez-vous et planification d'une tâche avec calcul des jours ouvrés via `BusinessCalendar`.

In [ ]:
from pycrmkit.calendars import BusinessCalendar
from pycrmkit.tasks import TaskPriority

contact_ref = EntityReference("contact", contact.id)

# 1. Enregistrement d'une activité
activity = crm.activities.log(
    type="meeting",
    subject="Comité scientifique annuel",
    direction="inbound",
    references=(contact_ref,),
)
print(f"📅 Activité consignée : [{activity.type}] {activity.subject}")

# 2. Calendrier Métier (exclusion des weekends)
calendar = BusinessCalendar(key="fr-business", timezone="Europe/Paris")
crm.calendars.register(calendar)

due_date = crm.calendars.due_in_business_days("fr-business", days=3)

# 3. Tâche planifiée
task = crm.tasks.create(
    title="Rédiger le compte-rendu de séance",
    priority=TaskPriority.HIGH,
    due_at=due_date,
    references=(contact_ref,),
)
print(f"📝 Tâche créée : {task.title} | Échéance dans 3 jours ouvrés : {task.due_at.strftime('%Y-%m-%d %H:%M %Z')}")

# Clôture de la tâche
completed_task = crm.tasks.complete(task.id)
print(f"   Statut de la tâche après complétion : {completed_task.status.value}")

## 6. Timeline Unifiée
La Timeline agrège automatiquement l'historique complet des interactions relatives au contact.

In [ ]:
timeline = crm.timeline.for_contact(contact.id)

print(f"📊 Timeline du contact {contact.display_name} ({timeline.total} entrées) :")
for entry in timeline.items:
    print(f"  • {entry.occurred_at.strftime('%H:%M:%S')} | [{entry.event_type}] {entry.title}")

## 7. Assertions de Validation du Notebook
Validation automatisée de toutes les propriétés du domaine Core CRM.

In [ ]:
assert org.id is not None
assert contact.emails[0].normalized == "marie.curie@institut-curie.org"
assert rel.is_primary is True
assert field_val.value == "Niveau Confidentiel Défense"
assert completed_task.status.value == "completed"
assert timeline.total >= 3

print("✨ TOUS LES TESTS DU NOTEBOOK 01 (CORE CRM) SONT VALIDÉS AVEC SUCCÈS !")